In [1]:
!pip install optuna-integration[xgboost]
#!rm -rf /kaggle/working/*


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 4.0 MB/s eta 0:00:00


In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import optuna
import xgboost as xgb
import warnings
import json

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

cv = KFold(n_splits = 5, shuffle = True, random_state = 42)

train = pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv", index_col =0)
test = pd.read_csv("/kaggle/input/playground-series-s6e1/test.csv", index_col =0)

working_dir = "/kaggle/working/"

for col in train.select_dtypes(include="object"):
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

target = "exam_score"

X = train.loc[:, train.columns != target]
y = train[target]
X_test = test

In [3]:
def objective(trial):

    params = {
    # Core
    "booster": "gbtree",
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "eval_metric": "rmse",

    # Learning
    "eta": trial.suggest_float("eta", 0.005, 0.3, log=True),

    # Tree complexity
    "max_depth": trial.suggest_int("max_depth", 3, 12),
    "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 20.0, log=True),
    "gamma": trial.suggest_float("gamma", 0.0, 10.0),
    "max_leaves": trial.suggest_int("max_leaves", 0, 256),

    # Sampling
    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
    "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
    "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 1.0),

    # Regularization
    "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),
    "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
    "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),

    # Categorical & missing
    "enable_categorical": True,

    # System
    "nthread": -1,
    "verbosity": 0,
    }

    rmses = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        dtrain = xgb.DMatrix(X_train, y_train, enable_categorical=True)
        dvalid = xgb.DMatrix(X_valid, y_valid, enable_categorical=True)

        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            evals=[(dvalid, "validation")],
            early_stopping_rounds=50,
            callbacks=[
                optuna.integration.XGBoostPruningCallback(
                    trial, "validation-rmse"
                )
            ],
            verbose_eval=False,
        )

        preds = booster.predict(dvalid)
        rmses.append(root_mean_squared_error(y_valid, preds))

    return float(np.mean(rmses))
    

In [4]:
study = optuna.create_study(
    direction="minimize",
    study_name="xgb_study",
    storage=f"sqlite:///{working_dir}optuna_xgb.db",
    load_if_exists=True,
)

study.optimize(objective, n_trials=100)

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)

best_params = study.best_params
with open(working_dir + "xgb_base_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-01-23 21:13:11,913] A new study created in RDB with name: xgb_study
[I 2026-01-23 21:19:53,277] Trial 0 finished with value: 8.757294360307958 and parameters: {'eta': 0.04868983671409779, 'max_depth': 12, 'min_child_weight': 0.500432006655084, 'gamma': 9.023445274749408, 'max_leaves': 143, 'subsample': 0.9979149465355748, 'colsample_bytree': 0.8250656620271841, 'colsample_bylevel': 0.9728927702592192, 'colsample_bynode': 0.9864760997551236, 'alpha': 6.549873402010993e-05, 'lambda': 3.308102832904116e-05, 'max_delta_step': 9}. Best is trial 0 with value: 8.757294360307958.
[I 2026-01-23 21:32:10,465] Trial 1 finished with value: 8.775914458141866 and parameters: {'eta': 0.03775346584650278, 'max_depth': 10, 'min_child_weight': 19.305422331242923, 'gamma': 4.556204852020769, 'max_leaves': 115, 'subsample': 0.5432718432442696, 'colsample_bytree': 0.7972658594332636, 'colsample_bylevel': 0.596487427995178, 'colsample_bynode': 0.755623203597489, 'alpha': 0.00015092357622531228, 'lam

Best params: {'eta': 0.04868983671409779, 'max_depth': 12, 'min_child_weight': 0.500432006655084, 'gamma': 9.023445274749408, 'max_leaves': 143, 'subsample': 0.9979149465355748, 'colsample_bytree': 0.8250656620271841, 'colsample_bylevel': 0.9728927702592192, 'colsample_bynode': 0.9864760997551236, 'alpha': 6.549873402010993e-05, 'lambda': 3.308102832904116e-05, 'max_delta_step': 9}
Best AUC: 8.757294360307958
